# Error Analysis — CRED-trained SVM vs MMedLlama-3 vs Phi-4

Error analysis of three models on the 615-row CRED test set, sliced by prediction outcome (TP, TN, FP, FN). For each model we examine three dimensions of each outcome group: entity distance, sentence length, and lexical-cue presence.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', lambda x: f'{x:.3f}')
pd.set_option('display.width', 120)

MODELS   = [('CRED-trained SVM','Prediction'), ('MMedLlama-3','MedLlama_Prediction'), ('Phi4','Phi4_Prediction')]
OUTCOMES = ['TP','TN','FP','FN']

In [2]:
df = pd.read_csv('/data/users/nency/CRED_application/test_data_analysis_table.csv')
print(f'Rows: {len(df)}')
print(f"Label distribution: {df['Label'].value_counts().to_dict()}")
for name, col in MODELS:
    print(f"  {name:<9}({col}) predicted Yes={int((df[col]==1).sum()):>3}  No={int((df[col]==0).sum()):>3}")

Rows: 615
Label distribution: {0: 561, 1: 54}
  CRED-trained SVM(Prediction) predicted Yes= 31  No=584
  MMedLlama-3(MedLlama_Prediction) predicted Yes=458  No=157
  Phi4     (Phi4_Prediction) predicted Yes=546  No= 69


In [3]:
def outcome_label(label, pred):
    if label==1 and pred==1: return 'TP'
    if label==0 and pred==0: return 'TN'
    if label==0 and pred==1: return 'FP'
    if label==1 and pred==0: return 'FN'

for name, col in MODELS:
    df[f'{name}_out'] = df.apply(lambda r: outcome_label(r['Label'], r[col]), axis=1)

# Biomedical-causal & association synonym lists (generic English fillers removed)
CAUSAL_TERMS = [
    'causation','causative','cause','caused','causes','causing','causal','causa','causal-agent',
    'induce','induced','inducer','inducing','inducement','induction','inducive',
    'stimulate','stimulation','stimulus','developement'
]
ASSOC_TERMS = [
    'associate','associations','association','associable','associative','associatory','affiliate',
    'link','linked','linkage','link-up','correlate','correlated','correlation','correlative','correlativity',
    'relation','connect','connection','connective','connexion','colligation','tie-in'
]
import re as _re
def _pat(words):
    esc = sorted([_re.escape(w) for w in set(words)], key=len, reverse=True)
    return _re.compile(r'\b(?:' + '|'.join(esc) + r')\b', _re.IGNORECASE)
P_CAUSAL = _pat(CAUSAL_TERMS)
P_ASSOC  = _pat(ASSOC_TERMS)
df['has_causal'] = df['sentence'].astype(str).apply(lambda s: bool(P_CAUSAL.search(s)))
df['has_assoc']  = df['sentence'].astype(str).apply(lambda s: bool(P_ASSOC.search(s)))
print(f"rows with causal trigger: {int(df['has_causal'].sum())} ({df['has_causal'].mean()*100:.1f}%)")
print(f"rows with assoc  trigger: {int(df['has_assoc'].sum())} ({df['has_assoc'].mean()*100:.1f}%)")
print(f"rows with neither       : {int((~df['has_causal'] & ~df['has_assoc']).sum())}")

def confusion(label_s, pred_s):
    tp = int(((label_s==1) & (pred_s==1)).sum())
    tn = int(((label_s==0) & (pred_s==0)).sum())
    fp = int(((label_s==0) & (pred_s==1)).sum())
    fn = int(((label_s==1) & (pred_s==0)).sum())
    p  = tp/(tp+fp) if tp+fp else 0.0
    r  = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    spec = tn/(tn+fp) if tn+fp else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn)
    return {'TP':tp,'TN':tn,'FP':fp,'FN':fn,'P':p,'R':r,'F1':f1,'Acc':acc,'Spec':spec}

def numeric_outcome_table(df, model, num_col, bins, bin_labels):
    out_col = f'{model}_out'
    rows = []
    for o in OUTCOMES:
        g = df[df[out_col]==o]
        if len(g)==0:
            rec = {'outcome':o,'N':0,'mean':float('nan'),'median':float('nan'),'min':float('nan'),'max':float('nan')}
        else:
            rec = {'outcome':o,'N':len(g),'mean':g[num_col].mean(),'median':g[num_col].median(),
                   'min':int(g[num_col].min()),'max':int(g[num_col].max())}
        binned = pd.cut(g[num_col], bins=bins, labels=bin_labels, right=True, include_lowest=True)
        for lab in bin_labels:
            rec[lab] = int((binned==lab).sum())
        rows.append(rec)
    return pd.DataFrame(rows).set_index('outcome')

def lexical_outcome_table(df, model):
    out_col = f'{model}_out'
    rows = []
    for o in OUTCOMES:
        mask = (df[out_col]==o)
        n = int(mask.sum())
        nc = int((mask & df['has_causal']).sum())
        na = int((mask & df['has_assoc']).sum())
        nn = int((mask & ~df['has_causal'] & ~df['has_assoc']).sum())
        rows.append({'outcome':o,'N':n,
                     'with_causal_kw':nc,'pct_causal': nc/n*100 if n else 0.0,
                     'with_assoc_kw': na,'pct_assoc':  na/n*100 if n else 0.0,
                     'no_keyword':    nn,'pct_no_kw':  nn/n*100 if n else 0.0})
    return pd.DataFrame(rows).set_index('outcome')

DIST_BINS   = [-0.1, 15, 30, 1e9]
DIST_LABELS = ['0-15','16-30','>30']
LEN_BINS    = [0, 200, 300, 1e9]
LEN_LABELS  = ['<=200','201-300','>300']

rows with causal trigger: 421 (68.5%)
rows with assoc  trigger: 273 (44.4%)
rows with neither       : 119


## Headline metrics

In [4]:
rows = []
for name, col in MODELS:
    m = confusion(df['Label'], df[col])
    rows.append({'model':name, **m})
headline = pd.DataFrame(rows).set_index('model')
headline

,TP,TN,FP,FN,P,R,F1,Acc,Spec
model,,,,,,,,,
CRED-trained SVM,19,549,12,35,0.613,0.352,0.447,0.924,0.979
MMedLlama-3,46,149,412,8,0.100,0.852,0.180,0.317,0.266
Phi4,52,67,494,2,0.095,0.963,0.173,0.193,0.119


## CRED-trained SVM — entity distance by outcome

In [5]:
numeric_outcome_table(df, 'CRED-trained SVM', 'min_words_between', DIST_BINS, DIST_LABELS)

,N,mean,median,min,max,0-15,16-30,>30
outcome,,,,,,,,
TP,19,6.842,5.000,1,23,18,1,0
TN,549,45.495,32.000,0,221,158,106,285
FP,12,13.667,7.500,1,42,9,1,2
FN,35,15.143,8.000,1,172,23,11,1


## CRED-trained SVM — sentence length by outcome

In [6]:
numeric_outcome_table(df, 'CRED-trained SVM', 'sentence_length', LEN_BINS, LEN_LABELS)

,N,mean,median,min,max,<=200,201-300,>300
outcome,,,,,,,,
TP,19,221.000,228.000,125,374,7,10,2
TN,549,285.929,278.000,126,462,54,298,197
FP,12,227.500,229.000,132,273,1,11,0
FN,35,258.543,245.000,212,456,0,32,3


## CRED-trained SVM — lexical cues by outcome

In [7]:
lexical_outcome_table(df, 'CRED-trained SVM')

,N,with_causal_kw,pct_causal,with_assoc_kw,pct_assoc,no_keyword,pct_no_kw
outcome,,,,,,,
TP,19,13,68.421,8,42.105,5,26.316
TN,549,365,66.485,235,42.805,112,20.401
FP,12,11,91.667,5,41.667,1,8.333
FN,35,32,91.429,25,71.429,1,2.857


## MMedLlama-3 — entity distance by outcome

In [8]:
numeric_outcome_table(df, 'MMedLlama-3', 'min_words_between', DIST_BINS, DIST_LABELS)

,N,mean,median,min,max,0-15,16-30,>30
outcome,,,,,,,,
TP,46,13.174,6.500,1,172,33,12,1
TN,149,41.638,34.000,1,221,29,41,79
FP,412,45.964,31.000,0,220,138,66,208
FN,8,6.750,5.500,1,12,8,0,0


## MMedLlama-3 — sentence length by outcome

In [9]:
numeric_outcome_table(df, 'MMedLlama-3', 'sentence_length', LEN_BINS, LEN_LABELS)

,N,mean,median,min,max,<=200,201-300,>300
outcome,,,,,,,,
TP,46,238.261,245.000,125,374,6,37,3
TN,149,317.758,281.000,127,462,14,63,72
FP,412,272.716,276.500,126,462,41,246,125
FN,8,286.000,266.500,125,456,1,5,2


## MMedLlama-3 — lexical cues by outcome

In [10]:
lexical_outcome_table(df, 'MMedLlama-3')

,N,with_causal_kw,pct_causal,with_assoc_kw,pct_assoc,no_keyword,pct_no_kw
outcome,,,,,,,
TP,46,39,84.783,30,65.217,5,10.870
TN,149,103,69.128,62,41.611,24,16.107
FP,412,273,66.262,178,43.204,89,21.602
FN,8,6,75.000,3,37.500,1,12.500


## Phi-4 — entity distance by outcome

In [11]:
numeric_outcome_table(df, 'Phi4', 'min_words_between', DIST_BINS, DIST_LABELS)

,N,mean,median,min,max,0-15,16-30,>30
outcome,,,,,,,,
TP,52,12.365,6.500,1,172,39,12,1
TN,67,46.896,34.000,1,221,18,12,37
FP,494,44.532,31.000,0,220,149,95,250
FN,2,8.500,8.500,4,13,2,0,0


## Phi-4 — sentence length by outcome

In [12]:
numeric_outcome_table(df, 'Phi4', 'sentence_length', LEN_BINS, LEN_LABELS)

,N,mean,median,min,max,<=200,201-300,>300
outcome,,,,,,,,
TP,52,244.250,245.000,125,456,7,40,5
TN,67,284.060,278.000,214,376,0,56,11
FP,494,284.763,277.000,126,462,55,253,186
FN,2,273.500,273.500,269,278,0,2,0


## Phi-4 — lexical cues by outcome

In [13]:
lexical_outcome_table(df, 'Phi4')

,N,with_causal_kw,pct_causal,with_assoc_kw,pct_assoc,no_keyword,pct_no_kw
outcome,,,,,,,
TP,52,44,84.615,31,59.615,6,11.538
TN,67,30,44.776,38,56.716,10,14.925
FP,494,346,70.040,202,40.891,103,20.850
FN,2,1,50.000,2,100.000,0,0.000


## Brief summary

After running the cells above, the dominant failure mode of each model is visible from its outcome counts:

- **CRED-trained SVM** is the only model whose TN bucket dominates (549 of 561 negatives); FPs (12) and FNs (35) are small but real. The mean entity distance in SVM's TPs is 6.8 words versus 45.5 words in its TNs, showing the model fires only on close-range pairs.
- **MMedLlama-3** predicts Yes for 458 of 615 rows; its FP bucket (412) dominates while it keeps a non-trivial TN bucket (149). FP mean entity distance is 46 words — the model fires Yes even on far-apart pairs.
- **Phi-4** is more aggressively Yes-biased (546 Yes predictions): FP=494, TN=67, FN=2. The TN bucket has the lowest causal-keyword rate of any group (~45%), showing Phi-4 only says No when the abstract lacks causal vocabulary.

Across all three dimensions (entity distance, sentence length, lexical cues), the LLM FP buckets are not preferentially concentrated in any subgroup — they span all bins and both cue conditions. The LLM errors are not driven by long-distance or noisy contexts (as is the case for SVM), but by a global Yes-bias inherited from the prompt and base model.

## Mean-value summary (9 rows × 4 outcomes)

Three rows per model — one per analyzed dimension. Lexical cue is reported as the percentage of rows in the outcome group that contain a causal keyword (binary indicator → mean → %).

In [14]:
OUT_ORDER = ['TP','FP','FN','TN']
rows = []
for name, _ in MODELS:
    out_col = f'{name}_out'
    r1 = {'metric': f'{name} - mean entity distance (words)'}
    r2 = {'metric': f'{name} - mean sentence length (words)'}
    r3 = {'metric': f'{name} - has-causal-keyword (%)'}
    for o in OUT_ORDER:
        m = (df[out_col]==o); n = int(m.sum())
        r1[o] = round(df.loc[m,'min_words_between'].mean(), 2) if n else float('nan')
        r2[o] = round(df.loc[m,'sentence_length'].mean(),  2) if n else float('nan')
        r3[o] = round((df['has_causal'] & m).sum()/n*100, 2) if n else float('nan')
    rows += [r1, r2, r3]
summary = pd.DataFrame(rows).set_index('metric')[OUT_ORDER]
summary


,TP,FP,FN,TN
metric,,,,
CRED-trained SVM - mean entity distance (words),6.840,13.670,15.140,45.500
CRED-trained SVM - mean sentence length (words),221.000,227.500,258.540,285.930
CRED-trained SVM - has-causal-keyword (%),68.420,91.670,91.430,66.480
MMedLlama-3 - mean entity distance (words),13.170,45.960,6.750,41.640
MMedLlama-3 - mean sentence length (words),238.260,272.720,286.000,317.760
MMedLlama-3 - has-causal-keyword (%),84.780,66.260,75.000,69.130
Phi4 - mean entity distance (words),12.370,44.530,8.500,46.900
Phi4 - mean sentence length (words),244.250,284.760,273.500,284.060
Phi4 - has-causal-keyword (%),84.620,70.040,50.000,44.780


# Criteria-based performance breakdown (manuscript-style)

Within each bin / condition we report per-model Precision, Recall, and F1 side-by-side.

In [15]:
def bin_perf(df, bin_series, bin_labels):
    rows = []
    for lab in bin_labels:
        sub = df[bin_series==lab]
        rec = {'bin': lab, 'N': len(sub)}
        for name, col in MODELS:
            tp = int(((sub['Label']==1) & (sub[col]==1)).sum())
            tn = int(((sub['Label']==0) & (sub[col]==0)).sum())
            fp = int(((sub['Label']==0) & (sub[col]==1)).sum())
            fn = int(((sub['Label']==1) & (sub[col]==0)).sum())
            p  = tp/(tp+fp) if tp+fp else 0.0
            r  = tp/(tp+fn) if tp+fn else 0.0
            f1 = 2*p*r/(p+r) if p+r else 0.0
            rec[f'{name}_P']  = round(p, 3)
            rec[f'{name}_R']  = round(r, 3)
            rec[f'{name}_F1'] = round(f1, 3)
        rows.append(rec)
    return pd.DataFrame(rows).set_index('bin')

def cond_perf(df, cond_labels):
    rows = []
    for lab, mask in cond_labels:
        sub = df[mask]
        rec = {'condition': lab, 'N': len(sub)}
        for name, col in MODELS:
            tp = int(((sub['Label']==1) & (sub[col]==1)).sum())
            tn = int(((sub['Label']==0) & (sub[col]==0)).sum())
            fp = int(((sub['Label']==0) & (sub[col]==1)).sum())
            fn = int(((sub['Label']==1) & (sub[col]==0)).sum())
            p  = tp/(tp+fp) if tp+fp else 0.0
            r  = tp/(tp+fn) if tp+fn else 0.0
            f1 = 2*p*r/(p+r) if p+r else 0.0
            rec[f'{name}_P']  = round(p, 3)
            rec[f'{name}_R']  = round(r, 3)
            rec[f'{name}_F1'] = round(f1, 3)
        rows.append(rec)
    return pd.DataFrame(rows).set_index('condition')


## Criterion 1 - Entity distance: per-bin P/R/F1

In [16]:
dist_binned = pd.cut(df['min_words_between'], bins=DIST_BINS, labels=DIST_LABELS,
                     right=True, include_lowest=True)
bin_perf(df, dist_binned, DIST_LABELS)


,N,CRED-trained SVM_P,CRED-trained SVM_R,CRED-trained SVM_F1,MMedLlama-3_P,MMedLlama-3_R,MMedLlama-3_F1,Phi4_P,Phi4_R,Phi4_F1
bin,,,,,,,,,,
0-15,208,0.667,0.439,0.529,0.193,0.805,0.311,0.207,0.951,0.341
16-30,119,0.500,0.083,0.143,0.154,1.000,0.267,0.112,1.000,0.202
>30,288,0.000,0.000,0.000,0.005,1.000,0.010,0.004,1.000,0.008


## Criterion 2 - Sentence length: per-bin P/R/F1

In [17]:
len_binned = pd.cut(df['sentence_length'], bins=LEN_BINS, labels=LEN_LABELS,
                    right=True, include_lowest=True)
bin_perf(df, len_binned, LEN_LABELS)


,N,CRED-trained SVM_P,CRED-trained SVM_R,CRED-trained SVM_F1,MMedLlama-3_P,MMedLlama-3_R,MMedLlama-3_F1,Phi4_P,Phi4_R,Phi4_F1
bin,,,,,,,,,,
<=200,62,0.875,1.000,0.933,0.128,0.857,0.222,0.113,1.000,0.203
201-300,351,0.476,0.238,0.317,0.131,0.881,0.228,0.137,0.952,0.239
>300,202,1.000,0.400,0.571,0.023,0.600,0.045,0.026,1.000,0.051


## Criterion 3 - Causal keyword dependency

P/R/F1 for rows with vs without any causal trigger, plus the manuscript's spurious-correlation (FP-in-causal) and implicit-causality (FN-in-no-keyword) statistics per model.

In [18]:
cond_perf(df, [('with_causal_kw', df['has_causal']),
               ('no_causal_kw',  ~df['has_causal'])])


,N,CRED-trained SVM_P,CRED-trained SVM_R,CRED-trained SVM_F1,MMedLlama-3_P,MMedLlama-3_R,MMedLlama-3_F1,Phi4_P,Phi4_R,Phi4_F1
condition,,,,,,,,,,
with_causal_kw,421,0.542,0.289,0.377,0.125,0.867,0.218,0.113,0.978,0.202
no_causal_kw,194,0.857,0.667,0.750,0.048,0.778,0.090,0.051,0.889,0.097


In [19]:
no_kw = ~df['has_causal'] & ~df['has_assoc']
rows = []
for name, col in MODELS:
    fp_mask = (df['Label']==0) & (df[col]==1)
    fn_mask = (df['Label']==1) & (df[col]==0)
    n_fp = int(fp_mask.sum())
    n_fn = int(fn_mask.sum())
    fp_in_causal = int((fp_mask & df['has_causal']).sum())
    fn_in_no_kw  = int((fn_mask & no_kw).sum())
    rows.append({'model':name,
                 'n_FP':n_fp,
                 'FP_in_causal_kw':fp_in_causal,
                 'pct_FP_in_causal_kw': round(fp_in_causal/n_fp*100,2) if n_fp else 0.0,
                 'n_FN':n_fn,
                 'FN_in_no_kw':fn_in_no_kw,
                 'pct_FN_in_no_kw': round(fn_in_no_kw/n_fn*100,2) if n_fn else 0.0})
pd.DataFrame(rows).set_index('model')


,n_FP,FP_in_causal_kw,pct_FP_in_causal_kw,n_FN,FN_in_no_kw,pct_FN_in_no_kw
model,,,,,,
CRED-trained SVM,12,11,91.670,35,1,2.860
MMedLlama-3,412,273,66.260,8,1,12.500
Phi4,494,346,70.040,2,0,0.000


## Criterion 4 - Association keyword dependency

In [20]:
cond_perf(df, [('with_assoc_kw', df['has_assoc']),
               ('no_assoc_kw',  ~df['has_assoc'])])


,N,CRED-trained SVM_P,CRED-trained SVM_R,CRED-trained SVM_F1,MMedLlama-3_P,MMedLlama-3_R,MMedLlama-3_F1,Phi4_P,Phi4_R,Phi4_F1
condition,,,,,,,,,,
with_assoc_kw,273,0.615,0.242,0.348,0.144,0.909,0.249,0.133,0.939,0.233
no_assoc_kw,342,0.611,0.524,0.564,0.064,0.762,0.118,0.067,1.000,0.126


# Disagreement analysis — SVM vs each LLM

For each LLM (MedLlama, Phi-4) we isolate the rows where its prediction differs from SVM's, then split those rows into two mutually exclusive groups:

* **SVM correct** — SVM matches the ground-truth `Label`; LLM does not.
* **LLM correct** — LLM matches the ground-truth `Label`; SVM does not.

Within each group we break the rows down by entity-distance bin, sentence-length bin, and presence of a causal keyword.

In [21]:
def disagreement_breakdown(df, llm_name, llm_col):
    disagree = df[df['Prediction'] != df[llm_col]].copy()
    svm_correct = disagree[disagree['Prediction'] == disagree['Label']]
    llm_correct = disagree[disagree[llm_col]      == disagree['Label']]
    print(f'SVM vs {llm_name}: {len(disagree)} disagreement rows '
          f'(SVM correct: {len(svm_correct)}, {llm_name} correct: {len(llm_correct)})')
    out = {}
    # Entity distance
    a = svm_correct['dist_bin'].value_counts().reindex(DIST_LABELS, fill_value=0)
    b = llm_correct['dist_bin'].value_counts().reindex(DIST_LABELS, fill_value=0)
    tab = pd.DataFrame({'CRED-trained SVM_correct': a, f'{llm_name}_correct': b})
    tab['CRED-trained SVM_correct_%']         = (tab['CRED-trained SVM_correct']/max(1,tab['CRED-trained SVM_correct'].sum())*100).round(1)
    tab[f'{llm_name}_correct_%'] = (tab[f'{llm_name}_correct']/max(1,tab[f'{llm_name}_correct'].sum())*100).round(1)
    out['entity_distance'] = tab
    # Sentence length
    a = svm_correct['len_bin'].value_counts().reindex(LEN_LABELS, fill_value=0)
    b = llm_correct['len_bin'].value_counts().reindex(LEN_LABELS, fill_value=0)
    tab = pd.DataFrame({'CRED-trained SVM_correct': a, f'{llm_name}_correct': b})
    tab['CRED-trained SVM_correct_%']         = (tab['CRED-trained SVM_correct']/max(1,tab['CRED-trained SVM_correct'].sum())*100).round(1)
    tab[f'{llm_name}_correct_%'] = (tab[f'{llm_name}_correct']/max(1,tab[f'{llm_name}_correct'].sum())*100).round(1)
    out['sentence_length'] = tab
    # Causal keyword
    a = svm_correct['has_causal'].value_counts().reindex([True, False], fill_value=0)
    b = llm_correct['has_causal'].value_counts().reindex([True, False], fill_value=0)
    a.index = ['with_causal_kw','no_causal_kw']
    b.index = ['with_causal_kw','no_causal_kw']
    tab = pd.DataFrame({'CRED-trained SVM_correct': a, f'{llm_name}_correct': b})
    tab['CRED-trained SVM_correct_%']         = (tab['CRED-trained SVM_correct']/max(1,tab['CRED-trained SVM_correct'].sum())*100).round(1)
    tab[f'{llm_name}_correct_%'] = (tab[f'{llm_name}_correct']/max(1,tab[f'{llm_name}_correct'].sum())*100).round(1)
    out['causal_keyword'] = tab
    return out

# Materialize the bin columns used by the helper (consistent with the rest of the notebook)
df['dist_bin'] = pd.cut(df['min_words_between'], bins=DIST_BINS, labels=DIST_LABELS,
                        right=True, include_lowest=True)
df['len_bin']  = pd.cut(df['sentence_length'],   bins=LEN_BINS,  labels=LEN_LABELS,
                        right=True, include_lowest=True)


## CRED-trained CRED-trained SVM vs MMedLlama-3

In [22]:
ml = disagreement_breakdown(df, 'MMedLlama-3', 'MedLlama_Prediction')


SVM vs MMedLlama-3: 435 disagreement rows (SVM correct: 404, MMedLlama-3 correct: 31)


### MMedLlama-3 — entity distance

In [23]:
ml['entity_distance']


,CRED-trained SVM_correct,MMedLlama-3_correct,CRED-trained SVM_correct_%,MMedLlama-3_correct_%
dist_bin,,,,
0-15,133,19,32.900,61.300
16-30,65,11,16.100,35.500
>30,206,1,51.000,3.200


### MMedLlama-3 — sentence length

In [24]:
ml['sentence_length']


,CRED-trained SVM_correct,MMedLlama-3_correct,CRED-trained SVM_correct_%,MMedLlama-3_correct_%
len_bin,,,,
<=200,41,0,10.100,0.000
201-300,238,30,58.900,96.800
>300,125,1,30.900,3.200


### MMedLlama-3 — causal keyword

In [25]:
ml['causal_keyword']


,CRED-trained SVM_correct,MMedLlama-3_correct,CRED-trained SVM_correct_%,MMedLlama-3_correct_%
with_causal_kw,265,29,65.600,93.500
no_causal_kw,139,2,34.400,6.500


## CRED-trained CRED-trained SVM vs Phi-4

In [26]:
ph = disagreement_breakdown(df, 'Phi4', 'Phi4_Prediction')


SVM vs Phi4: 517 disagreement rows (SVM correct: 483, Phi4 correct: 34)


### Phi-4 — entity distance

In [27]:
ph['entity_distance']


,CRED-trained SVM_correct,Phi4_correct,CRED-trained SVM_correct_%,Phi4_correct_%
dist_bin,,,,
0-15,141,22,29.200,64.700
16-30,94,11,19.500,32.400
>30,248,1,51.300,2.900


### Phi-4 — sentence length

In [28]:
ph['sentence_length']


,CRED-trained SVM_correct,Phi4_correct,CRED-trained SVM_correct_%,Phi4_correct_%
len_bin,,,,
<=200,54,0,11.200,0.000
201-300,243,31,50.300,91.200
>300,186,3,38.500,8.800


### Phi-4 — causal keyword

In [29]:
ph['causal_keyword']


,CRED-trained SVM_correct,Phi4_correct,CRED-trained SVM_correct_%,Phi4_correct_%
with_causal_kw,336,32,69.600,94.100
no_causal_kw,147,2,30.400,5.900


## Brief summary

* SVM wins ~13× more often than either LLM when they disagree (404 vs 31 for MedLlama; 483 vs 34 for Phi-4).
* SVM's unique strength is the **>30-word entity-distance bin** — about half of its winning rows have entities >30 words apart, where the LLM almost always wrongly says Yes.
* The LLM's unique strength is the **0-15 word bin** — close-by entities in causal-sounding text, where SVM is too conservative and misses the positive.
* Across both LLMs, **~94% of LLM-wins occur in causal-keyword sentences** (vs corpus baseline 68.5%), confirming the LLM is reading the lexical trigger and getting lucky when the trigger genuinely applies to the target pair.
* SVM wins are spread across all sentence-length bins; LLM wins concentrate in the 201-300 bucket (>90%) and are essentially absent on short or long abstracts.